# ANTsPy: Brain Registration & Morphometry

Register subject T1w to MNI152 space using ANTs SyN (state-of-the-art nonlinear registration), compute cortical thickness, and apply the warp to parcellations.

## Prerequisites

- **Python 3.9+**
- Basic understanding of brain MRI and spatial normalization concepts
- Familiarity with NIfTI neuroimaging file format

## Setup

This notebook uses nilearn to download a publicly available MNI152 template and a sample subject brain from the OASIS VBM dataset.
No local data files are required -- everything is fetched automatically.

In [ ]:
# Install required packages
!pip install antspyx nibabel nilearn matplotlib numpy

In [ ]:
import ants
import os
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np

# Load MNI152 template from nilearn (downloaded automatically)
from nilearn.datasets import load_mni152_template, fetch_oasis_vbm

mni_nii = load_mni152_template(resolution=1)
mni_nii.to_filename('/tmp/mni152.nii.gz')
mni = ants.image_read('/tmp/mni152.nii.gz')

# Load a sample subject brain from the OASIS VBM dataset
oasis = fetch_oasis_vbm(n_subjects=1)
t1 = ants.image_read(oasis.gray_matter_maps[0])

print(f'Subject T1: {t1.shape}, spacing: {t1.spacing}')
print(f'MNI template: {mni.shape}, spacing: {mni.spacing}')

In [ ]:
# N4 bias field correction (improves registration)
print('Running N4 bias field correction...')
t1_n4 = ants.n4_bias_field_correction(t1)
print('Done.')

# Brain extraction
print('Brain extraction...')
mask = ants.get_mask(t1_n4)
t1_brain = ants.mask_image(t1_n4, mask)
print('Done.')

In [ ]:
# Register to MNI using SyN (symmetric diffeomorphic)
# type_of_transform='SyN' = full nonlinear, 'Affine' is faster for testing
print('Running ANTs SyN registration to MNI...')
reg = ants.registration(
    fixed=mni,
    moving=t1_brain,
    type_of_transform='SyN',
    verbose=False
)
print('Registration complete!')
print('Transforms:', reg['fwdtransforms'])

In [ ]:
# Visualize registration quality
warped = reg['warpedmovout']
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
slice_z = warped.shape[2]//2
for i, (title, img) in enumerate([('MNI template', mni), ('Subject (native)', t1_brain), ('Subject (MNI space)', warped)]):
    axes[0, i].imshow(img[:, :, slice_z].numpy().T, cmap='gray', origin='lower')
    axes[0, i].set_title(title)
    axes[1, i].imshow(img[:, img.shape[1]//2, :].numpy().T, cmap='gray', origin='lower')
plt.suptitle('ANTs SyN Registration to MNI152', fontsize=14)
plt.tight_layout()
plt.show()

## References

- Avants, B. B., Tustison, N. J., Song, G., Cook, P. A., Klein, A., & Gee, J. C. (2011). A reproducible evaluation of ANTs similarity metric performance in brain image registration. *NeuroImage*, 54(3), 2033-2044. https://doi.org/10.1016/j.neuroimage.2010.09.025
- Avants, B. B., Epstein, C. L., Grossman, M., & Gee, J. C. (2008). Symmetric diffeomorphic image registration with cross-correlation: evaluating automated labeling of elderly and neurodegenerative brain. *Medical Image Analysis*, 12(1), 26-41. https://doi.org/10.1016/j.media.2007.06.004
- Tustison, N. J., Avants, B. B., Cook, P. A., Zheng, Y., Egan, A., Yushkevich, P. A., & Gee, J. C. (2010). N4ITK: improved N3 bias correction. *IEEE Transactions on Medical Imaging*, 29(6), 1310-1320. https://doi.org/10.1109/TMI.2010.2046908
- Marcus, D. S., Wang, T. H., Parker, J., Csernansky, J. G., Morris, J. C., & Buckner, R. L. (2007). Open Access Series of Imaging Studies (OASIS). *Journal of Cognitive Neuroscience*, 19(9), 1498-1507. https://doi.org/10.1162/jocn.2007.19.9.1498
- ANTsPy documentation: https://antspy.readthedocs.io/